In [43]:
!which python

/home/user/jfayzullaev/stellar-clustering/.venv-vis/bin/python


In [44]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import normalized_mutual_info_score as NMI, adjusted_rand_score as ARI


In [45]:
EMB = 'LINE_res'

In [46]:
KMEANS_FILE_TX   = f"{EMB}/transactions_line_kmeans_results.csv"


In [47]:
NORM_LABELS_PATH = os.path.expanduser("~/stellar-clustering/publication/labeled-data/normalization/labels_mapped_normalized.csv")

In [48]:
N_SPLITS     = 5
RANDOM_STATE = 42

In [49]:
def overall_purity_comm(df_comm_name: pd.DataFrame, comm_col: str, name_col: str = "name") -> float:

    if df_comm_name.empty:
        return np.nan
    counts = df_comm_name.groupby([comm_col, name_col]).size().reset_index(name='cnt')
    totals = counts.groupby(comm_col)['cnt'].sum()
    max_per_comm = counts.groupby(comm_col)['cnt'].max()
    return float(max_per_comm.sum() / totals.sum())


In [50]:
def load_kmeans_fixed_k(kmeans_file: str, k_col: str) -> pd.DataFrame:

    if not os.path.exists(kmeans_file):
        raise FileNotFoundError(f"no file: {kmeans_file}")
    df = pd.read_csv(kmeans_file)

    if df.columns[0] != "account_id":
        print(f"Renaming first column '{df.columns[0]}' to 'account_id'")
        df.rename(columns={df.columns[0]: "account_id"}, inplace=True)


    df = df[['account_id', k_col]].dropna().drop_duplicates()
    df = df.rename(columns={k_col: 'cluster'})


    try:
        df['account_id'] = df['account_id'].astype(int)
    except Exception:
        df['account_id'] = df['account_id'].astype(str)
    return df

In [51]:
def evaluate_fixed_kmeans_cv(
    labels_path: str,
    clusters_df: pd.DataFrame,
    label_col: str = "name",
    n_splits: int = 5,
    random_state: int = 42
):



    labels = (pd.read_csv(labels_path)
                .dropna(subset=['account_id', label_col])
                .drop_duplicates(subset=['account_id'])
                .rename(columns={label_col: 'name'}))

    # sync dtype
    try:
        labels['account_id'] = labels['account_id'].astype(int)
        clu = clusters_df.copy()
        clu['account_id'] = clu['account_id'].astype(int)
    except Exception:
        labels['account_id'] = labels['account_id'].astype(str)
        clu = clusters_df.copy()
        clu['account_id'] = clu['account_id'].astype(str)

    joined = labels.merge(clu, on='account_id', how='inner')

    n_labeled = len(labels)
    n_joined  = len(joined)
    coverage  = (n_joined / n_labeled) if n_labeled else 0.0
    if n_joined == 0:
        raise ValueError("No labeled accounts found in this K-Means file after join.")

    le = LabelEncoder()
    y_all = le.fit_transform(joined['name'].values)
    X_ids = joined['account_id'].values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_ids, y_all), start=1):
        df_tr = joined.iloc[tr_idx].copy()
        df_te = joined.iloc[te_idx].copy()

        y_true_tr = le.transform(df_tr['name'])
        y_pred_tr = df_tr['cluster'].values
        nmi_tr = NMI(y_true_tr, y_pred_tr)
        ari_tr = ARI(y_true_tr, y_pred_tr)
        purity_tr = overall_purity_comm(df_tr[['cluster', 'name']], comm_col='cluster', name_col='name')

        y_true_te = le.transform(df_te['name'])
        y_pred_te = df_te['cluster'].values
        nmi_te = NMI(y_true_te, y_pred_te)
        ari_te = ARI(y_true_te, y_pred_te)
        purity_te = overall_purity_comm(df_te[['cluster', 'name']], comm_col='cluster', name_col='name')

        rows.append({
            'fold': fold,
            'n_train': len(df_tr),
            'n_test': len(df_te),
            'train_frac': len(df_tr) / len(joined),
            'NMI_train': nmi_tr,
            'ARI_train': ari_tr,
            'Purity_train': purity_tr,
            'NMI_test': nmi_te,
            'ARI_test': ari_te,
            'Purity_test': purity_te,
        })

    per_fold_df = pd.DataFrame(rows)

    averages = {
        'Avg_NMI_train': float(np.nanmean(per_fold_df['NMI_train'].values)),
        'Avg_ARI_train': float(np.nanmean(per_fold_df['ARI_train'].values)),
        'Avg_Purity_train': float(np.nanmean(per_fold_df['Purity_train'].values)),
        'Avg_NMI_test': float(np.nanmean(per_fold_df['NMI_test'].values)),
        'Avg_ARI_test': float(np.nanmean(per_fold_df['ARI_test'].values)),
        'Avg_Purity_test': float(np.nanmean(per_fold_df['Purity_test'].values)),
        'Avg_train_frac': float(np.nanmean(per_fold_df['train_frac'].values)),
    }

    coverage_info = {
        'n_labeled': int(n_labeled),
        'n_joined': int(n_joined),
        'coverage': float(coverage),
    }

    return per_fold_df, averages, coverage_info



## Transactions

In [52]:
K_COL = "kmeans_10" 

In [53]:
clusters_df = load_kmeans_fixed_k(KMEANS_FILE_TX, K_COL)

Renaming first column 'node_id' to 'account_id'


In [54]:
norm_labels_path = os.path.expanduser(NORM_LABELS_PATH)

In [55]:
norm_per_fold, norm_avg, norm_cov = evaluate_fixed_kmeans_cv(
    labels_path=norm_labels_path,
    clusters_df=clusters_df,
    label_col="name",
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE
)

/home/user/jfayzullaev/stellar-clustering/.venv-vis/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [56]:
print(f"\n K-Means {K_COL} | Normalized Labels per fold metrics")
display(norm_per_fold)


 K-Means kmeans_10 | Normalized Labels per fold metrics


,fold,n_train,n_test,train_frac,NMI_train,ARI_train,Purity_train,NMI_test,ARI_test,Purity_test
0,1,6668,1668,0.799904,0.080283,0.010017,0.935213,0.096971,0.016881,0.935252
1,2,6669,1667,0.800024,0.082729,0.013495,0.934923,0.087382,0.002985,0.937013
2,3,6669,1667,0.800024,0.082205,0.011154,0.935073,0.088102,0.011314,0.935813
3,4,6669,1667,0.800024,0.079512,0.009934,0.935373,0.100449,0.017154,0.934613
4,5,6669,1667,0.800024,0.078326,0.011835,0.935523,0.102051,0.008828,0.934013


In [57]:
print(f"\n K-Means {K_COL} | Normalized Labels avgs")
for k, v in norm_avg.items():
    print(f"{k}: {v:.6f}")
print(f"Coverage: {norm_cov['coverage']:.2%}  ({norm_cov['n_joined']}/{norm_cov['n_labeled']})")



 K-Means kmeans_10 | Normalized Labels avgs
Avg_NMI_train: 0.080611
Avg_ARI_train: 0.011287
Avg_Purity_train: 0.935221
Avg_NMI_test: 0.094991
Avg_ARI_test: 0.011432
Avg_Purity_test: 0.935341
Avg_train_frac: 0.800000
Coverage: 100.00%  (8336/8336)


In [58]:
TX_PATH = f'{EMB}/cross-validation'

In [59]:
os.makedirs(TX_PATH, exist_ok=True)

In [60]:
norm_per_fold.to_csv(f"{TX_PATH}/tx_cv_{K_COL}_norm_per_fold.csv", index=False)
pd.DataFrame([{
    **norm_avg,
    **norm_cov,
    "k_col": K_COL,
    "source_file": os.path.basename(KMEANS_FILE_TX)
}]).to_csv(f"{TX_PATH}/tx_cv_{K_COL}_norm_summary.csv", index=False)
print(f"Saved to {TX_PATH}")

Saved to LINE_res/cross-validation
